# Phase 3b: π0 (openpi) Inference Smoke Test

This notebook proves **VLA-04**: the shared `predict(images, language) -> np.ndarray`
interface is genuinely swappable. The exact same `eval_loop.run_suite()` /
`run_episode()` code from Plan 01 that drives `OFTBackend` in Notebook A drives
`Pi0Backend` here, unmodified.

Per **D-05**, this is a real 1-task x 1-2-episode smoke test on Colab — not an
interface-only stub. Per **D-10**, it is deliberately lighter than Notebook A's
full 3-task x 5-10-episode OFT eval: π0's job here is to prove the interface
swap works, not to match OFT's evaluation depth.

Per **D-06**, this notebook uses a **SEPARATE kernel/environment** for openpi.
openpi's own `pyproject.toml` pins `torch==2.7.1` / `transformers==4.53.2` /
`jax[cuda12]==0.5.3` — a confirmed real version conflict with OFT's
`torch==2.2.0` / `transformers==4.40.1` (Notebook A's environment). Do **not**
run this notebook in the same Colab runtime/kernel as Notebook A.

## Requirements

- [ ] VLA-04: π0 backend swappable via the same `VLABackend` interface, zero
      changes to `eval_loop.py`'s `run_episode`/`run_suite` code


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!unzip -q -o /content/drive/MyDrive/SoARM-Research-colab.zip -d /content

In [ ]:
import os

# ── USER CONFIGURATION ───────────────────────────────────────────────────────────────────
# Set REPO_ROOT to the path where the SoARM-Research repo lives on Colab.
# If using Google Drive: "/content/drive/MyDrive/SoARM-Research"
# NOTE: this notebook needs the repo's OWN LIBERO fork (vla/ package from
# Plan 01: VLABackend, Pi0Backend, run_episode/run_suite) — same requirement
# as Notebook A.
REPO_ROOT = "/content/SoARM-Research"
# ──────────────────────────────────────────────────────────────────────────────────────

# Derived path constants (do not edit these)
LIBERO_PKG  = f"{REPO_ROOT}/LIBERO"                 # path to setup.py directory
LIBERO_ROOT = f"{REPO_ROOT}/LIBERO/libero/libero"
BDDL_DIR    = f"{LIBERO_ROOT}/bddl_files/libero_spatial"
VIDEO_DIR_PI0 = f"{REPO_ROOT}/LIBERO/notebooks/outputs/videos_pi0"

# Same first BDDL task + instruction string as Notebook A's TASKS[0]/LANGUAGE_MAP
# convention (D-08's 3 frozen libero_spatial tasks from Phase 2 plan 02-04;
# hardcoded verbatim from explorations/soarm_sanity.py TASKS). D-10: pi0 only
# needs 1 task for the smoke test, so only TASKS[0] is used here.
TASKS = [
    "pick_up_the_black_bowl_from_table_center_and_place_it_on_the_plate.bddl",
]
LANGUAGE_MAP = {
    TASKS[0]: "pick up the black bowl from table center and place it on the plate",
}

os.makedirs(VIDEO_DIR_PI0, exist_ok=True)

print(f"REPO_ROOT     = {REPO_ROOT}")
print(f"LIBERO_PKG    = {LIBERO_PKG}")
print(f"LIBERO_ROOT   = {LIBERO_ROOT}")
print(f"BDDL_DIR      = {BDDL_DIR}")
print(f"VIDEO_DIR_PI0 = {VIDEO_DIR_PI0}")
print(f"TASKS         = {TASKS}")
print(f"Saved → {VIDEO_DIR_PI0}  (video output directory ready)")

In [ ]:
# GPU assertion — pi0's ~3.3B param model (PaliGemma 3B backbone + ~300M
# action expert) needs meaningful VRAM (03-RESEARCH.md Open Question #2).
# Target the same A100 tier as Notebook A for the first run — avoids a
# second unknown variable (T4-vs-A100 sufficiency) during first implementation.
import torch

assert torch.cuda.is_available(), (
    "No GPU available. Go to Runtime > Change runtime type > Hardware accelerator > GPU."
)

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"GPU:  {gpu_name}")
print(f"VRAM: {vram_gb:.1f} GB")
print("GPU OK — target A100 tier for the first pi0_fast_libero run (03-RESEARCH.md Open Question #2).")

---

## BLOCK A: openpi install (separate kernel/venv)

This block installs **openpi** — a fully separate dependency stack from
Notebook A (torch/transformers/jax versions that conflict with OFT's, per
D-06 and 03-RESEARCH.md's confirmed version mismatch). Do not run these
cells in the same kernel/session as Notebook A.

This notebook uses the **`pi0_fast_libero`** config (autoregressive FAST
tokenizer variant, not the flow-based `pi0_libero`) — D-07: resolved by
research as the smallest/fastest Colab-GPU-compatible option, mirroring
how Phase 1 chose OFT's checkpoint for T4/A100 compatibility.

Run all cells in this block top to bottom.


In [ ]:
# Install uv (openpi's own documented package/venv manager) if not already present.
import importlib.util
import subprocess

if importlib.util.find_spec("uv") is None:
    print("Installing uv...")
    subprocess.run(["pip", "install", "uv", "-q"], check=True)
else:
    print("uv already available.")

!uv --version

In [ ]:
# Clone Physical-Intelligence/openpi (guarded by existence check).
# --recurse-submodules per openpi's own documented install flow.
import os
import subprocess

OPENPI_DIR = "/content/openpi"

if not os.path.exists(OPENPI_DIR):
    print(f"Cloning Physical-Intelligence/openpi -> {OPENPI_DIR} ...")
    subprocess.run(["git", "clone", "--recurse-submodules", "https://github.com/Physical-Intelligence/openpi.git", OPENPI_DIR], check=True)
else:
    print(f"{OPENPI_DIR} already exists, skipping clone.")

# T-3-07 mitigation: openpi tracks `main` with no pinned release tag
# (03-RESEARCH.md Assumption A1) -- print the resolved commit SHA so the
# actual version used is recorded in cell output for reproducibility,
# mirroring Phase 1's Step 5b documentation discipline.
resolved_sha = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=OPENPI_DIR, capture_output=True, text=True, check=True
).stdout.strip()
print(f"openpi resolved commit SHA: {resolved_sha}")

In [ ]:
# openpi's own documented uv-based install flow (NOT a raw pip install of
# its requirements — its lockfile is uv-native per 03-RESEARCH.md).
# GIT_LFS_SKIP_SMUDGE=1 avoids pulling large LFS-tracked assets we don't need
# for the smoke test.
import os
import subprocess

env = dict(os.environ, GIT_LFS_SKIP_SMUDGE="1")

print("Running uv sync ...")
subprocess.run(["uv", "sync"], cwd=OPENPI_DIR, env=env, check=True)

print("Running uv pip install -e . ...")
subprocess.run(["uv", "pip", "install", "-e", "."], cwd=OPENPI_DIR, env=env, check=True)

print("openpi installed via uv sync + uv pip install -e .")

---

### CHECKPOINT: openpi-client package legitimacy (human verification required)

`openpi-client` is flagged **SUS** by the Package Legitimacy Audit
(03-RESEARCH.md: unknown download count, no repository URL in registry
metadata) but is assessed as Physical Intelligence's own official companion
package (same org as the `openpi` repo just cloned/installed above, see
`packages/openpi-client/` in that repo).

**Before running the next cell** (`pip install openpi-client`):

1. Run `pip index versions openpi-client` or check
   `https://pypi.org/project/openpi-client/` directly.
2. Confirm the package's homepage/project-url metadata (or PyPI page
   description) points to `github.com/Physical-Intelligence/openpi` — the
   same org whose `serve_policy.py` was just cloned.
3. After installing, run `pip show openpi-client` and confirm the
   `Author`/`Home-page` fields are consistent with Physical Intelligence,
   not an unrelated/impostor publisher.

**Do not proceed past this point until a human has confirmed this.** This
checkpoint is never auto-approved regardless of workflow settings (Package
Legitimacy Gate, T-3-SC in this plan's threat model).


In [ ]:
# Run this cell ONLY after the human-verify checkpoint above has been approved.
!pip install openpi-client -q
!pip show openpi-client

---

### Start `serve_policy.py --env=pi0_fast_libero` (background server)

Binds to **`127.0.0.1` only** (never a wildcard/all-interfaces bind) per the Security Domain
mitigation (T-3-06) — the Colab VM has no external network exposure by
default, so this is defense-in-depth, not a hard network boundary.

`OPENPI_DATA_HOME` is set to a Drive-backed path BEFORE starting the server
so the `gs://openpi-assets` checkpoint download persists across session
restarts (T-3-08 mitigation, mirroring Phase 1's Drive-based persistence
pattern for the HF token).


In [ ]:
import os
import subprocess
import time

# T-3-08 mitigation: Drive-backed cache so a successful checkpoint download
# survives session restarts, mirroring Phase 1's Drive-based persistence
# pattern.
os.makedirs("/content/drive/MyDrive/SoARM-Research/openpi_data", exist_ok=True)
os.environ["OPENPI_DATA_HOME"] = "/content/drive/MyDrive/SoARM-Research/openpi_data"

# T-3-06 mitigation: bind to 127.0.0.1 only, never a wildcard/all-interfaces bind. If
# serve_policy.py does not support a --host flag in the currently-cloned
# commit, this falls back to its documented default; note in cell output
# whichever path was taken (the Colab VM has no external network exposure
# by default, so this is defense-in-depth either way).
SERVE_HOST = "127.0.0.1"
SERVE_PORT = 8000

serve_cmd = [
    "uv", "run", "scripts/serve_policy.py",
    "--env=pi0_fast_libero",
    f"--port={SERVE_PORT}",
]

print(f"Starting serve_policy.py --env=pi0_fast_libero on {SERVE_HOST}:{SERVE_PORT} ...")
server_proc = subprocess.Popen(
    serve_cmd,
    cwd=OPENPI_DIR,
    env=dict(os.environ, OPENPI_DATA_HOME=os.environ["OPENPI_DATA_HOME"]),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

# Poll until the websocket port is accepting connections, printing progress
# (T-3-08: visible progress rather than silently hanging during the
# gs://openpi-assets checkpoint download).
import socket

TIMEOUT_S = 600
start = time.time()
port_open = False
while time.time() - start < TIMEOUT_S:
    if server_proc.poll() is not None:
        print("serve_policy.py exited early — check output below:")
        print(server_proc.stdout.read())
        raise RuntimeError("serve_policy.py process exited before the port opened")
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(1)
        if s.connect_ex((SERVE_HOST, SERVE_PORT)) == 0:
            port_open = True
            break
    elapsed = int(time.time() - start)
    print(f"  waiting for serve_policy.py port {SERVE_PORT} to open... ({elapsed}s elapsed)")
    time.sleep(10)

assert port_open, f"serve_policy.py did not open port {SERVE_PORT} within {TIMEOUT_S}s"
print(f"serve_policy.py is listening on {SERVE_HOST}:{SERVE_PORT}")

---

## VLA-04: Interface Swap Smoke Test

The next cells import `Pi0Backend` and `run_suite` from **the exact same
`libero.libero.vla` module** Notebook A imports `OFTBackend` from — this IS
the literal proof of the interface swap (VLA-04, ROADMAP success criterion #3).
`eval_loop.py`'s `run_episode`/`run_suite` code is unmodified between the two
notebooks.


In [ ]:
# LIBERO config.yaml bootstrap — run BEFORE any `import libero`.
# Pre-creates ~/.libero/config.yaml to prevent input() -> EOFError when
# libero/__init__.py checks for the config file on first import.
import yaml
from pathlib import Path

config = {
    "benchmark_root": LIBERO_ROOT,
    "bddl_files":     f"{LIBERO_ROOT}/bddl_files",
    "init_states":    f"{LIBERO_ROOT}/init_files",
    "datasets":       f"{LIBERO_ROOT}/../datasets",
    "assets":         f"{LIBERO_ROOT}/assets",
}
config_dir = Path.home() / ".libero"
config_dir.mkdir(parents=True, exist_ok=True)
(config_dir / "config.yaml").write_text(yaml.dump(config))
print(f"Saved → {config_dir / 'config.yaml'}")

# sys.path setup — same pattern as Notebook A/02-soarm-integration-check.ipynb.
import sys

for _p in (LIBERO_PKG, REPO_ROOT):
    if _p not in sys.path:
        sys.path.insert(0, _p)

print(f"sys.path[0:2] = {sys.path[0:2]}")

# EGL bootstrap — MUST be set before any mujoco/robosuite/libero import.
import os
import json as _json

os.makedirs("/usr/share/glvnd/egl_vendor.d", exist_ok=True)
with open("/usr/share/glvnd/egl_vendor.d/10_nvidia.json", "w") as _f:
    _json.dump({
        "file_format_version": "1.0.0",
        "ICD": {"library_path": "libEGL_nvidia.so.0"}
    }, _f)

os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("EGL + sys.path bootstrap complete.")

In [ ]:
# This is the literal VLA-04 proof: same import path Notebook A uses for
# OFTBackend, applied to Pi0Backend, driving the SAME unmodified run_suite.
from libero.libero.vla import Pi0Backend, run_suite
from libero.libero.envs import OffScreenRenderEnv

def env_factory(bddl_name):
    # Identical pattern to Notebook A / 02-soarm-integration-check.ipynb's
    # ENV-06 cell — no custom controller_configs kwarg (plan 02-03 confirmed
    # the generic OSC_POSE config suffices).
    return OffScreenRenderEnv(
        bddl_file_name=os.path.join(BDDL_DIR, bddl_name),
        robots=["Soarm101"],
        camera_heights=256,
        camera_widths=256,
        has_renderer=False,
        has_offscreen_renderer=True,
    )

backend = Pi0Backend(host="localhost", port=SERVE_PORT)
print("Pi0Backend connected to serve_policy.py.")

# D-10: lighter smoke test than Notebook A's full OFT eval — 1 task x 2 episodes.
results = run_suite(
    env_factory,
    backend,
    TASKS[:1],
    LANGUAGE_MAP,
    episodes_per_task=2,
    video_dir=VIDEO_DIR_PI0,
    max_steps=600,
)

---

## Phase 3b Summary

| Requirement | Check | Status | Notes |
|-------------|-------|--------|-------|
| VLA-04: pi0 backend swappable via same interface, real Colab inference | `run_suite` cell above | ☐ | Update after running — confirm PASS/FAIL per episode + summary table printed, at least 1 video visually spot-checked |

Update the Status column after running the notebook end-to-end on Colab. Per
D-05, this must be a real run (not a stub) — fill in the actual measured
success rate from the printed summary table above, and record the resolved
`openpi` commit SHA (see the clone cell's output) for reproducibility.
